 # Libraries

In [1]:
!pip install ijson --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.4/134.4 kB 3.9 MB/s eta 0:00:00


In [20]:
import os
import json
import ijson
import pandas as pd
from tqdm import tqdm
from typing import Generator,Dict
from collections import defaultdict

In [37]:
tqdm.pandas()

In [5]:
PATH = "/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json"

# Data Loading

In [16]:
class IJSONLoader:

    def __init__(self, filename : str) -> None:
        self.filename = filename

    def __iter__(self) -> Generator[Dict, None, None]:

        total_bytes = os.path.getsize(self.filename)

        with tqdm(total=total_bytes) as pbar:

            with open(self.filename, "r") as f:
    
                for line in f:
                    
                    bytes_read = len(line)
    
                    try:
                        pbar.update(bytes_read)
                        yield json.loads(line.strip('\n'))
                    except Exception as e:
                        print(f"Unexpected error occured : {e}")

In [17]:
records = list(IJSONLoader(PATH))

100%|█████████▉| 99.99999999999906/100 [02:09<00:00,  1.30s/it]  


In [43]:
df = pd.DataFrame(records)

- Filter by year

In [44]:
df['year'] = df['update_date'].progress_apply(lambda x : int(x.split('-')[0]))

100%|██████████| 2895350/2895350 [00:03<00:00, 787588.08it/s]


In [45]:
df = df[df['year'] >= 2012]

- Get unique categories

In [46]:
unique_categories = defaultdict(int)

for record in tqdm(records):
    for cat in record['categories'].split():
        unique_categories[cat] += 1

100%|██████████| 2895350/2895350 [00:03<00:00, 956245.18it/s] 


In [47]:
cs_categories = { cat : count for cat,count in unique_categories.items() if cat.startswith('cs.') }
print(cs_categories)

{'cs.CG': 7698, 'cs.IT': 52137, 'cs.NE': 16811, 'cs.AI': 152693, 'cs.DS': 27500, 'cs.CE': 9577, 'cs.MS': 2513, 'cs.NA': 31712, 'cs.CC': 12241, 'cs.DM': 15038, 'cs.LO': 18310, 'cs.CR': 43991, 'cs.NI': 25953, 'cs.LG': 243308, 'cs.PF': 4872, 'cs.SE': 23442, 'cs.AR': 7237, 'cs.SC': 2953, 'cs.CY': 25726, 'cs.IR': 22989, 'cs.CV': 174689, 'cs.OH': 2289, 'cs.DB': 10794, 'cs.DL': 5678, 'cs.HC': 26804, 'cs.PL': 9236, 'cs.GT': 13711, 'cs.DC': 26593, 'cs.MA': 10668, 'cs.CL': 97670, 'cs.MM': 8810, 'cs.RO': 47154, 'cs.ET': 6518, 'cs.GL': 226, 'cs.FL': 5691, 'cs.OS': 1200, 'cs.SD': 19208, 'cs.GR': 8414, 'cs.SY': 41838, 'cs.SI': 22544}


In [48]:
sum(cs_categories.values())

1286436

- Select a subset from computer science

In [52]:
mask = df['categories'].progress_apply(lambda x : any([(cat in cs_categories.keys()) for cat in x.split(' ')]))
df = df[mask]

100%|██████████| 2334158/2334158 [00:03<00:00, 689042.61it/s]


In [54]:
len(df)

818560

In [55]:
subset = df.sample(8000)

In [56]:
subset.to_csv("subset.csv", index=False)

In [4]:
from datetime import datetime

datetime.now().strftime("%H:%M:%S")

'18:16:06'